# TP4 : Implémentation d'un système RAG (Retrieval-Augmented Generation)

**Étudiant :** ELAALGI Hatim — Module Système Distribué Agentique, Master S3

## Présentation du TP

Ce travaux pratiques (TP) est dédié à la mise en place et à l'évaluation d'une architecture **RAG** (*Retrieval-Augmented Generation*). L'objectif principal est de pallier les limites de connaissances d'un grand modèle de langage (LLM) en lui permettant d'extraire et de s'appuyer sur des informations issues de documents externes spécifiques avant de générer une réponse.

Au cours de ce TP, nous allons aborder les étapes clés suivantes :
* **Le Document Loading & Splitting** : Chargement et découpage des documents en blocs (*chunks*).
* **L'Indexation & Vectorisation** : Transformation des textes en vecteurs et stockage dans une base de données vectorielle.
* **Le Retrieval (Recherche)** : Extraction des segments de texte les plus pertinents vis-à-vis d'une requête utilisateur.
* **La Génération & Évaluation** : Injection du contexte dans le prompt du LLM et analyse de la fidélité de ses réponses.

## Mes choix

| Élément | Notebook du prof | Ce notebook |
|---|---|---|
| Document | `pdfs/nc346.pdf` (note de conjoncture DEPF) | **mon CV** `pdfs/CV_ELAALGI_Hatim_2026_FR_Global.pdf`, comme le propose le prof |
| Découpage | `RecursiveCharacterTextSplitter`, 300 tokens, overlap 20 | identique |
| Embeddings | OpenAI `text-embedding-3-small` | **FastEmbed** local `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensions, multilingue, 0,2 Go) |
| Base vectorielle | Chroma, `./store` | identique |
| LLM du RAG | OpenAI `gpt-4o-mini` | Groq `qwen/qwen3.8-27b` |
| LLM juge | OpenAI `gpt-4o` (plus puissant que le LLM du RAG) | Groq `openai/gpt-oss-120b` (120 milliards de paramètres, contre 27 pour le RAG) |

Groq ne fournit pas de modèle d'embedding : les vecteurs sont calculés **localement** dans le conteneur avec FastEmbed (modèles ONNX, sans PyTorch). Comme mon CV est en français, j'ai choisi un modèle **multilingue**, et léger, car il est téléchargé dans le conteneur.

# 0. Preparation de l'environement 

Dans le terminal `uv init` , ` uv venv `, ` Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process ` et ` .venv\Scripts\activate ` .  ` uv add ipykernel ` et  ` uv add langchain ` puis ` uv add langchain-openai ` et  `uv add langchain-text-splitters ` puis  `uv add langchain-community ` finalement pour importer la clee api open ai ` uv add python-dotenv  `

> **Dans mon dépôt :** même environnement uv conteneurisé que pour les TP précédents. J'ai ajouté `uv add langchain-community langchain-text-splitters pypdf chromadb fastembed` (LangChain, `langchain-groq` et `python-dotenv` étaient déjà là depuis le TP3), puis reconstruit l'image. La clé `GROQ_API_KEY` est dans le fichier `.env`, qui n'est pas versionné.

In [1]:
import json
import os
import shutil
import warnings
warnings.filterwarnings("ignore", message=".*IProgress.*")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import tiktoken
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
from IPython.display import Markdown, display

/tmp/ipykernel_7/3798989971.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader


##  LLM

In [2]:
load_dotenv(override=True)
print("GROQ_API_KEY trouvée :", bool(os.getenv("GROQ_API_KEY")))

GROQ_API_KEY trouvée : True


In [3]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0, max_retries=8)

# 1. Loading the pdf, Chunking

Le prof place `nc346.pdf` dans un dossier `pdfs` et propose de choisir son CV à la place. J'ai donc mis mon CV (2 pages) dans `pdfs/`.

In [4]:
pdf_file = "./pdfs/CV_ELAALGI_Hatim_2026_FR_Global.pdf"

In [5]:
pdf_loader=PyPDFLoader(pdf_file)

Et si vous voulez charger un dossier qui contient plusieurs fichiers, vous utilisez `PyPDFDirectoryLoader(path="./pdfs")`.

On découpe avec `RecursiveCharacterTextSplitter` et l'encodeur `tiktoken` : des blocs (*chunks*) de 300 tokens au plus, avec un chevauchement (*overlap*) de 20 tokens. Le découpage a besoin de `pypdf` (`uv add pypdf`).

In [6]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

In [7]:
chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)

12

In [8]:
# Ce que contient chaque chunk : page, taille en tokens et début du texte
enc = tiktoken.get_encoding("o200k_base")
pd.set_option("display.max_colwidth", 90)
pd.DataFrame([{"page": c.metadata["page_label"],
               "tokens": len(enc.encode(c.page_content)),
               "début du chunk": c.page_content[:90].replace("\n", " ")} for c in chunks])

,page,tokens,début du chunk
0,1,283,PROFIL Étudiant\ten\t Master\t2\tIntelligence\tArtificielle\t&\tDigital\tComputing \te...
1,1,257,Développeur\tOdoo\tERP Juillet\t–\tNovembre\t2025 Dynamic\tHorizon\t ·\tTanger Chatbot...
2,1,272,2026 Kafka\t\t·\t\tFastAPI\t\t·\t\tDocker\t\t·\t\tTensorFlow\t\t·\t\tXGBoost\t\t·\t\tO...
3,1,274,Architecture \t:\tfrontière\tde\tconfiance\timposée\tpar\tle\tcode\tentre\tun\tmodèle\...
4,1,290,interroge\tdonc\tles\tvraies\ttables\t;\tAPI\tREST\tFlask\tsécurisée. Tableaux\tde\tbo...
5,1,268,automatique\t;\tlip-sync\tde\tl'avatar\tcalculé\ten\ttemps\tréel. Pipeline\ttemps\trée...
6,1,33,"modèle\tstatique\t;\tlimites\tdocumentées\t(F1\tde\t0,28\tsur\tles\tclasses\tles\tplus..."
7,2,264,"Détection\tde\tfraude\tbancaire\tnon\tsupervisée\t—\tVAE,\tLSTM-AE,\tIsolation\tForest..."
8,2,264,"LLM\t&\tIA\tgénérative Agents\tLLM\tà\tgarde-fous,\tRAG\t(ChromaDB,\tFAISS,\tSentence-..."
9,2,288,"Sécurité OIDC\t/\tSSO,\tcontrôle\td'accès,\tjournalisation\td'audit,\tfiltrage\tdes\ti..."


**Ce qu'on observe.** Mon CV (2 pages) donne **12 chunks**, presque tous entre 257 et 291 tokens, donc juste sous la limite de 300. Le chunk n° 6 n'a que 33 tokens : c'est la fin de la page 1, car le découpage ne mélange pas deux pages. Un document long comme la note de conjoncture du prof en produit beaucoup plus.

Deux défauts de l'extraction PDF apparaissent déjà dans l'aperçu :
- PyPDF sépare les mots par des **tabulations** (`Étudiant\ten\tMaster`) au lieu d'espaces ;
- l'**ordre du texte** extrait ne suit pas toujours l'ordre visuel du CV. Par exemple, le titre du projet NIDS-FL est à la fin du chunk 1, alors que sa description (« FedAvg sur 3 clients ») se retrouve dans le chunk 5, après la fin du projet Ibn Battuta. La section 9 montre la conséquence.

# 2.vectore Store - ChromaDB et Embedding

*Importer le model d'embedding*

Code du prof (OpenAI, payant) :
```python
from langchain_openai import OpenAIEmbeddings
embedding_model=OpenAIEmbeddings(model='text-embedding-3-small')
```

In [9]:
from langchain_community.embeddings import FastEmbedEmbeddings
embedding_model = FastEmbedEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

v = embedding_model.embed_query("apprentissage fédéré")
print("dimension :", len(v), "| premières valeurs :", [round(x, 3) for x in v[:5]])

/app/.venv/lib/python3.14/site-packages/huggingface_hub/utils/tqdm.py:176: UserWarning: Cannot enable progress bars: environment variable `HF_HUB_DISABLE_PROGRESS_BARS=1` is set and has priority.
  warnings.warn(


Error while downloading from https://huggingface.co/Qdrant/paraphrase-multilingual-MiniLM-L12-v2-onnx-Q/resolve/faf4aa4225822f3bc6376869cb1164e8e3feedd0/tokenizer.json: The read operation timed out
Trying to resume download...


Error while downloading from https://huggingface.co/Qdrant/paraphrase-multilingual-MiniLM-L12-v2-onnx-Q/resolve/faf4aa4225822f3bc6376869cb1164e8e3feedd0/model_optimized.onnx: The read operation timed out
Trying to resume download...


Error while downloading from https://huggingface.co/Qdrant/paraphrase-multilingual-MiniLM-L12-v2-onnx-Q/resolve/faf4aa4225822f3bc6376869cb1164e8e3feedd0/tokenizer.json: [Errno -3] Temporary failure in name resolution
Trying to resume download...


Error while downloading from https://huggingface.co/Qdrant/paraphrase-multilingual-MiniLM-L12-v2-onnx-Q/resolve/faf4aa4225822f3bc6376869cb1164e8e3feedd0/model_optimized.onnx: [Errno -3] Temporary failure in name resolution
Trying to resume download...


Error while downloading from https://huggingface.co/Qdrant/paraphrase-multilingual-MiniLM-L12-v2-onnx-Q/resolve/faf4aa4225822f3bc6376869cb1164e8e3feedd0/tokenizer.json: [Errno -3] Temporary failure in name resolution
Trying to resume download...


Error while downloading from https://huggingface.co/Qdrant/paraphrase-multilingual-MiniLM-L12-v2-onnx-Q/resolve/faf4aa4225822f3bc6376869cb1164e8e3feedd0/model_optimized.onnx: [Errno -3] Temporary failure in name resolution
Trying to resume download...


dimension : 384 | premières valeurs : [0.321, 0.087, -0.217, -0.009, -0.048]


Petit test : deux formulations du même sujet doivent avoir des vecteurs plus proches qu'une phrase sans rapport.

In [10]:
import numpy as np

def cosinus(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

q = embedding_model.embed_query("Quelles langues parle-t-il ?")
for phrase in ["Arabe langue maternelle, français courant, anglais intermédiaire",
               "Pipeline temps réel avec Kafka et FastAPI",
               "La recette du couscous aux sept légumes"]:
    print(f"{cosinus(q, embedding_model.embed_query(phrase)):+.3f}  {phrase}")

+0.480  Arabe langue maternelle, français courant, anglais intermédiaire


+0.058  Pipeline temps réel avec Kafka et FastAPI
+0.156  La recette du couscous aux sept légumes


Ajouter un dossier "store" et utiliser la commande ` uv add chromadb ` pour ajouter *chromadb*

Je vectorise les chunks et je les stocke dans la base vectorielle **vectorstore**. Je vide `./store` avant : sinon, chaque nouvelle exécution du notebook ajouterait les mêmes chunks en double dans la collection.

In [11]:
shutil.rmtree("./store", ignore_errors=True)
vectorstore=Chroma.from_documents(chunks,embedding_model,collection_name="cv_elaalgi",persist_directory="./store")
print("vecteurs dans la collection :", vectorstore._collection.count())

vecteurs dans la collection : 12


# 3.  **Trouver les chunks liée à la questions (Retriever)**

Le code du prof est :
```python
retriever=vectorstore.as_retriever(search_type='similarity',search_Kwargs={'k',5})
```
Le commentaire indique que l'on veut les **5** chunks les plus proches. Mais `search_Kwargs` (avec un K majuscule) n'est pas un paramètre reconnu, et `{'k',5}` est un **ensemble**, pas un dictionnaire. Le paramètre est donc ignoré et le retriever garde sa valeur par défaut. Vérification :

In [12]:
retriever_prof = vectorstore.as_retriever(search_type='similarity', search_Kwargs={'k',5})
print("code du prof :", len(retriever_prof.invoke("Quels sont ses projets ?")), "chunks renvoyés")

#crée le Retriever : k=5, les cinq chunks les plus proches de la question
retriever=vectorstore.as_retriever(search_type='similarity',search_kwargs={'k':5})
print("corrigé      :", len(retriever.invoke("Quels sont ses projets ?")), "chunks renvoyés")

code du prof : 4 chunks renvoyés
corrigé      : 5 chunks renvoyés


Test avec la question :
> « Chez quelle entreprise travaille-t-il actuellement ? »

In [13]:
#test
retrieved_chunks=retriever.invoke("Chez quelle entreprise travaille-t-il actuellement ?")
print("nombre de chunks est :",len(retrieved_chunks))
for d in retrieved_chunks:
    print(f"\n--- page {d.metadata['page_label']} ---\n{d.page_content[:250]} ...")

nombre de chunks est : 5

--- page 1 ---
PROFIL
Étudiant	en	
Master	2	Intelligence	Artificielle	&	Digital	Computing
	et	développeur	IA	chez	
Synertic
	
depuis	avril	2026,	je	conçois	des
systèmes	de	bout	en	bout	:	
agents	LLM
	encadrés	par	des	garde-fous,	
modèles	de	machine	learning
	évalué ...

--- page 2 ---
LLM	&	IA	générative
Agents	LLM	à	garde-fous,	RAG	(ChromaDB,	FAISS,	Sentence-Transformers),	modèles	vision-langage	locaux	(LM
Studio),	NL	→	SQL,	LangChain,	Google	Gemini
Évaluation
AUC-PR,	calibrage	de	seuil,	détection	de	fuites,	validation	par	sujet, ...

--- page 2 ---
Sécurité
OIDC	/	SSO,	contrôle	d'accès,	journalisation	d'audit,	filtrage	des	injections	de	prompt,	frontières	de	confiance	entre
modèles
FORMATION
Master	—	Intelligence	Artificielle	&	Digital	Computing
FST,	Université	Moulay	Slimane		|		2025	–	2027
Ma ...

--- page 2 ---
ResearchWatch
	 —	 moteur	 de	 recherche	 bilingue	 sous	 Elasticsearch	 sur	 500	 articles	 arXiv	 :	 précision@10	 de	 0,57,	 MRR	 de	 0,88,
das

# 4.  *Prompt Design*

## 4.1 Prompt template

In [14]:
prompt_template="""
Answer the following question based only on provided context The context is about the CV (resume)
of ELAALGI Hatim, a Master student in Artificial Intelligence & Digital Computing
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

### Note sur le contexte
Comme le conseille le prof, j'ai remplacé la description de la note de la DEPF par celle de **mon document** : le CV d'ELAALGI Hatim. Le modèle sait ainsi que « il » dans les questions désigne la personne du CV.

### Gestion des hallucinations
La consigne *« If the answer is not found in the context, answer: Je ne sais pas! »* oblige le modèle à avouer qu'il ne sait pas au lieu d'inventer.

## 4.2 *Retrieving the Relevant Documents*

Je récupère les chunks pertinents, je garde uniquement leur texte (`d.page_content`), puis je les concatène avec le séparateur `". "`.

In [15]:
user_input="Quel était le sujet de son projet de fin d'études de Licence ?"

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

print(len(context_for_query), "caractères de contexte")

3360 caractères de contexte


In [16]:
len(relevant_document_chunks)

5

## 4.3 Le prompt

On injecte le contexte et la question dans le template. Le prompt contient :
 1. le message système : ( Answer the following question ............  Je ne sais pas! ) ;
 2. le contexte : entre `<context>` et `</context>` ;
 3. la question : entre `<question>` et `</question>`.

In [17]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt[:900], "\n[...]\n", prompt[-250:])


Answer the following question based only on provided context The context is about the CV (resume)
of ELAALGI Hatim, a Master student in Artificial Intelligence & Digital Computing
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
Sécurité
OIDC	/	SSO,	contrôle	d'accès,	journalisation	d'audit,	filtrage	des	injections	de	prompt,	frontières	de	confiance	entre
modèles
FORMATION
Master	—	Intelligence	Artificielle	&	Digital	Computing
FST,	Université	Moulay	Slimane		|		2025	–	2027
Master	2	en	cours		·		diplôme	prévu	en	juin	2027
Licence	—	Analyse	de	Données
FST	Tanger,	Université	Abdelmalek	Essaâdi		|		2022	–	2025
Baccalauréat	—	Sciences	Physiques
2020
LANGUES
Arabe
Langue	maternelle
Français
Courant,	usage	professionnel
Anglais
Intermédiaire	:	communication	à	l'oral	et	à	l'écri 
[...]
 ncidents	 de	 compromission	 sur	 hôtes	 Linux	 —	 corrélation	 des	 évènements,
cartograph

# 5. *Récupérer la réponse de la part de llm*

In [18]:
resp=llm.invoke(prompt)
display(Markdown(resp.content))

Le sujet de son projet de fin d'études de Licence était le **Chatbot ATLAS — Plateforme conversationnelle d'analyse statistique**.

## 6. Regroupement dans une application RAG

### Définition de la fonction RAG
`RAG(query, llm, prompt_template)` enchaîne les trois étapes :
1. **le retriever** cherche les chunks proches de `query` et garde leur texte ;
2. **la concaténation** de ces textes forme le contexte ;
3. **la génération** : le contexte et la question sont injectés dans le template, puis on appelle `llm.invoke()`.

In [19]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

### Appel de RAG
D'abord la question du prof, qui n'a **aucun rapport** avec mon CV :

In [20]:
resp=RAG(" qu'ils sont les événements culturels dans le sud-est de Maroc ? ")
display(Markdown(resp))

Je ne sais pas!

Puis une question dont la réponse est dans le CV :

In [21]:
resp=RAG("Quels résultats obtient-il avec le VAE dans son projet de détection de fraude ?")
display(Markdown(resp))

Selon le contexte fourni, les résultats obtenus avec le VAE dans le projet de détection de fraude bancaire sont les suivants :

*   **AUC-PR** : Le VAE atteint un score de **0,1139**, contre 0,0497 pour l'Isolation Forest, soit une amélioration de **+129 %**.
*   **Rappel (Recall)** : Le VAE obtient un rappel de **44,2 %**, contre 26,9 % pour l'Isolation Forest.
*   **Coût total** : Grâce à un seuil calibré sur le coût métier, le coût total est réduit de **24 %**.

# 7. Évaluation de la réponse du LLM

## 7.1 Architecture

![Architecture RAG + LLM juge](./rag_juge.gif)

Le RAG (LLM1) génère une réponse. Le **LLM juge** reçoit la question, le contexte retrouvé et cette réponse, puis renvoie un **score**.

In [22]:
#question ou query
user_input="Chez quelle entreprise travaille-t-il actuellement et sur quel projet ?"

Récupérer le context 

In [23]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [24]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [25]:
# Réponse d'un RAG
answer=RAG(user_input)
display(Markdown(answer))

Il travaille actuellement chez **Synertic** sur le projet de **Copilote d'investigation de sécurité à base d'agents LLM**.

## 7.2- *Groundness*

Le prompt pour le juge 

In [26]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [27]:
# Concernant le juge, il est nécessaire d'employer un modèle plus puissant que celui mis en œuvre par le RAG.
groundness_checker=ChatGroq(
      model="openai/gpt-oss-120b",
      temperature=0,
      max_retries=8
)

In [28]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question)
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response=model.invoke(prompt)
    return juge_response.content

In [29]:
#évaluer
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
display(Markdown(resp))

**Étapes pour évaluer la réponse selon la métrique « la réponse doit être dérivée uniquement des informations présentées dans le contexte »**

1. **Identifier la question**  
   - Repérer les deux informations demandées :  
     a) le nom de l’entreprise où il travaille actuellement ;  
     b) le projet sur lequel il travaille actuellement.

2. **Extraire les informations pertinentes du contexte**  
   - Chercher dans le texte du contexte les passages qui mentionnent son emploi actuel et le projet associé.  
   - Noter exactement les libellés utilisés dans le contexte (sans reformulation libre).

3. **Comparer la réponse générée avec les extraits du contexte**  
   - Vérifier que chaque élément de la réponse (entreprise, projet) apparaît textuellement dans le contexte.  
   - S’assurer qu’aucune information supplémentaire ou différente n’est introduite.

4. **Détecter d’éventuelles dérivations ou inventions**  
   - Si la réponse contient des détails qui ne figurent pas dans le contexte, la métrique n’est pas respectée.  
   - Si la réponse omet une partie de l’information demandée, la métrique est partiellement respectée.

5. **Attribuer le score**  
   - 1 : aucune conformité.  
   - 2 : conformité très limitée.  
   - 3 : bonne conformité mais avec quelques écarts.  
   - 4 : grande conformité, presque aucune dérive.  
   - 5 : conformité totale, la réponse se limite exactement aux faits du contexte.

---

**Application de ces étapes à l’exemple fourni**

1. **Question**  
   - « Chez quelle entreprise travaille‑t‑il actuellement et sur quel projet ? »

2. **Informations du contexte**  
   - Section *EXPÉRIENCE PROFESSIONNELLE* :  
     - « Développeur IA — Copilote d'investigation de sécurité à base d'agents LLM »  
     - Période : « Avril 2026 – présent »  
     - Entreprise : « Synertic »  
   - Le même libellé apparaît également dans la description du poste sous *Synertic*.

3. **Correspondance avec la réponse**  
   - Réponse : « Il travaille actuellement chez **Synertic** sur le projet de **Copilote d'investigation de sécurité à base d'agents LLM**. »  
   - Les deux éléments (nom de l’entreprise et intitulé du projet) sont exactement ceux du contexte, sans ajout ni modification.

4. **Vérification d’éventuelles dérivations**  
   - Aucun terme supplémentaire n’est présent.  
   - Aucun détail absent n’est requis par la question.

5. **Score**  
   - La réponse se limite strictement aux informations du contexte et répond pleinement à la question.  
   - **Score : 5** (la métrique est entièrement respectée).

### Échelle de notation pour la fidélité au contexte (Faithfulness)

| Score | Signification | Description |
| :--- | :--- | :--- |
| **5 / 5** | **Excellent (Parfait)** | La réponse utilise **uniquement** les informations du contexte. Aucune hallucination, aucune extrapolation. |
| **4 / 5** | **Bon** | La réponse est correcte et s'appuie sur le contexte, mais la formulation pourrait être légèrement plus précise. |
| **3 / 5** | **Moyen** | La réponse est partiellement vraie, mais elle omet un détail important du contexte ou manque de clarté. |
| **2 / 5** | **Médiocre** | La réponse contient des extrapolations ou des informations non vérifiables avec le contexte fourni. |
| **1 / 5** | **Mauvais (Hallucination)** | La réponse est totalement hors-sujet, fausse, ou invente des données qui ne sont pas du tout dans le contexte. |

## 8. Évaluation sur un jeu de questions

Une seule question ne suffit pas pour juger un RAG. J'ai donc écrit **9 questions** sur mon CV et, pour chacune, la réponse attendue (que je connais, puisque c'est mon CV) :
- 7 questions dont la réponse **est dans le CV**, sur des parties différentes (formation, expérience, projets, langues) ;
- 1 question **absente** du CV (date de naissance) ;
- 1 question **hors sujet** (la question du prof sur la note de conjoncture).

Pour chaque question, je mesure :
1. **retrieval** : le chunk qui contient la réponse est-il parmi les 5 chunks retrouvés ? On teste la présence d'un mot-clé de la réponse attendue dans le contexte. PyPDF sépare les mots de mon CV par des **tabulations** (`Digital\tComputing`) : je normalise donc les espaces avant de comparer ;
2. **fidélité** : la note du juge. Pour pouvoir l'extraire automatiquement, j'ajoute au prompt du juge une consigne de format pour la dernière ligne ;
3. **exactitude** : je compare moi-même la réponse à la réponse attendue (dans l'analyse). Le juge ne vérifie que la fidélité au contexte, pas la justesse.

La fonction `evaluate()` du prof appelle `RAG()` elle-même : le juge pourrait donc noter une autre réponse que celle affichée. Ici, `juger()` reçoit **la** réponse affichée et **le** contexte qui a servi à la produire.

In [30]:
import re

questions = [
    # (question, mot-clé qui doit être dans le contexte, réponse attendue)
    ("Dans quel master est-il inscrit et quand obtiendra-t-il son diplôme ?", "Digital Computing",
     "Master IA & Digital Computing, FST Université Moulay Slimane, diplôme prévu en juin 2027"),
    ("Chez quelle entreprise travaille-t-il actuellement et sur quel projet ?", "Synertic",
     "Synertic (ESN), copilote d'investigation de sécurité à base d'agents LLM"),
    ("Quel était le sujet de son projet de fin d'études de Licence ?", "ATLAS",
     "Chatbot ATLAS au HCP : NL vers SQL sur les bases statistiques, Flask, Gemini"),
    ("Quels résultats obtient-il avec le VAE dans son projet de détection de fraude ?", "0,1139",
     "AUC-PR 0,1139 contre 0,0497 (+129 %), rappel 44,2 %, coût réduit de 24 %"),
    ("Combien de clients participent à l'apprentissage fédéré du projet NIDS-FL ?", "FedAvg",
     "3 clients (FedAvg)"),
    ("Quelles langues parle-t-il et à quel niveau ?", "maternelle",
     "arabe (maternelle), français (courant), anglais (intermédiaire)"),
    ("Quelle technologie utilise-t-il pour l'avatar 3D du projet Ibn Battuta ?", "Three.js",
     "Three.js (avec FastAPI, React 19, Qwen3-VL-4B)"),
    ("Quelle est sa date de naissance ?", None, "absente du CV : Je ne sais pas"),
    ("QU'il est la capitalisation boursière de Maroc pour 2025 ?", None, "hors sujet : Je ne sais pas"),
]

format_score = "\nTerminez votre réponse par une dernière ligne au format exact : Score : X/5"


def extraire_score(jugement):
    notes = re.findall(r"Score\W{0,6}([1-5])", jugement, flags=re.IGNORECASE)
    return int(notes[-1]) if notes else None


def normaliser(texte):
    return " ".join(texte.split())   # tabulations et retours à la ligne -> un espace


def juger(question, contexte, reponse, model=groundness_checker):
    prompt = f"""
     {groundness_rater_system_message + format_score}\n
     USER:
     {user_message_template.format(question=question, context=contexte, answer=reponse)}
     """
    return model.invoke(prompt).content


lignes, jugements = [], {}
for question, mot_cle, attendu in questions:
    contexte = ". ".join(d.page_content for d in retriever.invoke(question))
    reponse = RAG(question)
    jugements[question] = juger(question, contexte, reponse)
    lignes.append({
        "question": question,
        "chunk utile retrouvé": "—" if mot_cle is None else ("oui" if mot_cle in normaliser(contexte) else "NON"),
        "réponse du RAG": reponse.strip(),
        "réponse attendue": attendu,
        "fidélité (juge)": extraire_score(jugements[question]),
    })

resultats = pd.DataFrame(lignes)
pd.set_option("display.max_colwidth", 220)
resultats

,question,chunk utile retrouvé,réponse du RAG,réponse attendue,fidélité (juge)
0,Dans quel master est-il inscrit et quand obtiendra-t-il son diplôme ?,oui,Il est inscrit en Master Intelligence Artificielle & Digital Computing et obtiendra son diplôme en juin 2027.,"Master IA & Digital Computing, FST Université Moulay Slimane, diplôme prévu en juin 2027",5
1,Chez quelle entreprise travaille-t-il actuellement et sur quel projet ?,oui,Il travaille actuellement chez **Synertic** sur le projet de **Copilote d'investigation de sécurité à base d'agents LLM**.,"Synertic (ESN), copilote d'investigation de sécurité à base d'agents LLM",5
2,Quel était le sujet de son projet de fin d'études de Licence ?,oui,Le sujet de son projet de fin d'études de Licence était le **Chatbot ATLAS — Plateforme conversationnelle d'analyse statistique**.,"Chatbot ATLAS au HCP : NL vers SQL sur les bases statistiques, Flask, Gemini",1
3,Quels résultats obtient-il avec le VAE dans son projet de détection de fraude ?,oui,"Selon le contexte fourni, les résultats obtenus avec le VAE dans le projet de détection de fraude bancaire sont les suivants :\n\n* **AUC-PR** : 0,1139 (contre 0,0497 pour l'Isolation Forest, soit une amélioration ...","AUC-PR 0,1139 contre 0,0497 (+129 %), rappel 44,2 %, coût réduit de 24 %",5
4,Combien de clients participent à l'apprentissage fédéré du projet NIDS-FL ?,NON,Je ne sais pas!,3 clients (FedAvg),1
5,Quelles langues parle-t-il et à quel niveau ?,oui,"D'après le contexte fourni, ELAALGI Hatim parle les langues suivantes aux niveaux indiqués :\n\n* **Arabe** : Langue maternelle\n* **Français** : Courant, usage professionnel\n* **Anglais** : Intermédiaire (com...","arabe (maternelle), français (courant), anglais (intermédiaire)",5
6,Quelle technologie utilise-t-il pour l'avatar 3D du projet Ibn Battuta ?,oui,Il utilise **Three.js** pour l'avatar 3D du projet Ibn Battuta.,"Three.js (avec FastAPI, React 19, Qwen3-VL-4B)",5
7,Quelle est sa date de naissance ?,—,Je ne sais pas!,absente du CV : Je ne sais pas,5
8,QU'il est la capitalisation boursière de Maroc pour 2025 ?,—,Je ne sais pas!,hors sujet : Je ne sais pas,2


Justification du juge (dernières lignes) pour les réponses notées moins de 5 :

In [31]:
for q, j in jugements.items():
    if (extraire_score(j) or 0) < 5:
        print(f"### {q}\n" + "\n".join(j.strip().splitlines()[-12:]) + "\n")

### Quel était le sujet de son projet de fin d'études de Licence ?

4. **Conformité**  
   - La réponse ne se base pas uniquement sur les informations du contexte concernant le projet de fin d'études de Licence.  
   - Elle ajoute un titre de projet qui n’est pas présent dans la section pertinente du contexte.  
   - Par conséquent, la métrique n’est **pas respectée**.

5. **Notation**  
   - Conformité nulle ou quasi nulle → score **1/5**.

**Conclusion** : La réponse ne respecte pas la contrainte d’être dérivée uniquement du contexte fourni.

Score : 1/5

### Combien de clients participent à l'apprentissage fédéré du projet NIDS-FL ?

- Le contexte ne mentionne ni le projet « NIDS‑FL », ni l’apprentissage fédéré, ni le nombre de clients participants.  
- La réponse « Je ne sais pas ! » ne provient pas du contexte ; elle indique simplement une absence de connaissance.  
- Ainsi, la réponse n’est **pas dérivée** des informations du contexte.

**Évaluation**

- Conformité à la métrique 

### Analyse des résultats de la section 8

| Question | Bon chunk dans le top 5 ? | Réponse du RAG | Exacte ? | Fidélité (juge) |
|---|---|---|---|---|
| Master et date du diplôme | oui | Master IA & Digital Computing, juin 2027 | oui | 5 |
| Entreprise et projet actuels | oui | Synertic, copilote d'investigation de sécurité | oui | 5 |
| PFE de Licence | oui | Chatbot ATLAS | oui | **1** |
| Résultats du VAE | oui | AUC-PR 0,1139, rappel 44,2 %, coût −24 % | oui | 5 |
| Clients de NIDS-FL | **non** | « Je ne sais pas ! » | non (3 clients) | 1 |
| Langues | oui | arabe, français, anglais, avec les niveaux | oui | 5 |
| Avatar 3D d'Ibn Battuta | oui | Three.js | oui | 5 |
| Date de naissance (absente) | — | « Je ne sais pas ! » | oui | 5 |
| Capitalisation boursière (hors sujet) | — | « Je ne sais pas ! » | oui | 2 |

**1. Quand le bon chunk est retrouvé, le RAG répond juste.** Les 6 questions dont le chunk utile est dans le top 5 reçoivent une réponse exacte et complète, sans rien inventer : les chiffres du VAE sont recopiés exactement.

**2. Pas d'hallucination.** Pour la date de naissance (absente du CV) et pour la question du prof sur la capitalisation boursière (hors sujet), le modèle répond « Je ne sais pas ! » au lieu d'inventer. La consigne du prompt fonctionne.

**3. Le seul échec vient du retrieval, pas du LLM.** Pour NIDS-FL, le chunk qui contient « FedAvg sur 3 clients » n'est pas dans le top 5 (rang 7, voir la section 9). Le LLM n'a donc jamais vu l'information et répond honnêtement qu'il ne sait pas. Mauvais retrieval + bon LLM = pas de réponse.

**4. Le juge n'est pas fiable sur deux points.**
- **Faux négatif sur ATLAS** (1/5). La réponse est exacte, et le titre « Chatbot ATLAS — Plateforme conversationnelle d'analyse statistique » est bien dans le contexte. Le juge affirme pourtant que ce titre « n'est pas présent dans la section pertinente ». Avec les tabulations et l'ordre mélangé du texte extrait, le juge ne relie pas le titre à la ligne « Projet de fin d'études de Licence » qui le suit.
- **Notation incohérente de « Je ne sais pas ! »** : 5/5 pour la date de naissance, 2/5 pour la question hors sujet et 1/5 pour NIDS-FL, alors que c'est la même réponse dans les trois cas. La métrique (« la réponse doit être dérivée uniquement du contexte ») ne dit pas comment noter une **abstention**. Le juge l'interprète tantôt comme parfaitement fidèle (rien n'est inventé), tantôt comme non dérivée du contexte.

Ce juge mesure donc la **fidélité**, pas l'**exactitude**, et sa note dépend de la façon dont il interprète la métrique. Pour l'améliorer, il faudrait préciser dans son prompt qu'une abstention justifiée vaut 5, et ajouter une seconde métrique (par exemple la pertinence de la réponse par rapport à la question).

## 9. Combien de chunks faut-il donner au LLM ?

Le RAG ne peut répondre que si le chunk qui contient la réponse fait partie des **k** chunks retrouvés. Pour chaque question à réponse connue, je classe **tous** les chunks du CV par similarité avec la question, et je note le **rang** du premier chunk qui contient le mot-clé de la réponse. Je compare ensuite trois valeurs de k : 4 (ce que renvoie en réalité le code du prof, à cause de `search_Kwargs`), 5 (la valeur voulue) et tous les chunks. Cette expérience n'appelle aucun LLM.

In [32]:
def rang_du_bon_chunk(question, mot_cle):
    docs = vectorstore.similarity_search(question, k=vectorstore._collection.count())
    return next((r for r, d in enumerate(docs, start=1) if mot_cle in normaliser(d.page_content)), None)


diagnostic = pd.DataFrame([{"question": q, "mot-clé": m, "rang du bon chunk": rang_du_bon_chunk(q, m)}
                           for q, m, _ in questions if m is not None])
for k in (4, 5):
    diagnostic[f"retrouvé avec k={k}"] = diagnostic["rang du bon chunk"].le(k).map({True: "oui", False: "NON"})
diagnostic

,question,mot-clé,rang du bon chunk,retrouvé avec k=4,retrouvé avec k=5
0,Dans quel master est-il inscrit et quand obtiendra-t-il son diplôme ?,Digital Computing,1,oui,oui
1,Chez quelle entreprise travaille-t-il actuellement et sur quel projet ?,Synertic,1,oui,oui
2,Quel était le sujet de son projet de fin d'études de Licence ?,ATLAS,4,oui,oui
3,Quels résultats obtient-il avec le VAE dans son projet de détection de fraude ?,"0,1139",2,oui,oui
4,Combien de clients participent à l'apprentissage fédéré du projet NIDS-FL ?,FedAvg,7,NON,NON
5,Quelles langues parle-t-il et à quel niveau ?,maternelle,4,oui,oui
6,Quelle technologie utilise-t-il pour l'avatar 3D du projet Ibn Battuta ?,Three.js,2,oui,oui


In [33]:
r = diagnostic["rang du bon chunk"]
print(f"{len(chunks)} chunks dans la base")
for k in (1, 3, 4, 5, len(chunks)):
    print(f"k = {k:>2} : bon chunk retrouvé pour {(r <= k).sum()}/{len(r)} questions")
print(f"MRR (rang réciproque moyen) : {(1 / r).mean():.2f}")

12 chunks dans la base
k =  1 : bon chunk retrouvé pour 2/7 questions
k =  3 : bon chunk retrouvé pour 4/7 questions
k =  4 : bon chunk retrouvé pour 6/7 questions
k =  5 : bon chunk retrouvé pour 6/7 questions
k = 12 : bon chunk retrouvé pour 7/7 questions
MRR (rang réciproque moyen) : 0.52


**Ce qu'on observe.**
- Le bon chunk est au **rang 1** pour la formation et pour Synertic, aux rangs 2 à 4 pour le PFE, le VAE, les langues et Three.js, et au **rang 7** pour NIDS-FL. Le MRR vaut 0,52.
- Avec **k = 1**, seules 2 questions sur 7 auraient eu le bon chunk ; avec **k = 3**, 4 sur 7. Il faut **k = 4** pour en couvrir 6, car deux réponses (PFE, langues) arrivent exactement au rang 4.
- Ici, k = 4 (le code du prof, avec `search_Kwargs` ignoré) et k = 5 donnent le même résultat : aucun bon chunk n'est au rang 5. Le bug n'a donc pas d'effet sur ces questions, mais c'est un hasard.
- Pour NIDS-FL, il faudrait k = 7, soit plus de la moitié du CV. On enverrait alors au LLM presque tout le document, ce qui retire l'intérêt du retrieval.

**Pourquoi NIDS-FL est mal classé.** Le chunk qui contient « FedAvg sur 3 clients » ne contient ni « NIDS-FL » ni le titre du projet. Ceux-ci sont restés dans les chunks 1 et 2, à cause de l'ordre du texte extrait du PDF (section 1). Pour le modèle d'embedding, ce chunk parle de pipeline Kafka, de dérive des données et de la fin du projet Ibn Battuta : il ressemble peu à la question.

Le petit test de la section 2 montre aussi les limites de ce modèle léger (384 dimensions) : la question sur les langues est bien plus proche de la phrase sur les langues (0,48) que des deux autres, mais la recette du couscous (0,16) obtient un score **plus élevé** que la phrase sur Kafka (0,06), alors qu'aucune des deux n'a de rapport avec la question.

## 10. Conclusion

- Le pipeline du prof (chargement, découpage de 300 tokens, embeddings, Chroma, top-k, prompt avec « Je ne sais pas ! », juge) fonctionne sur mon CV. Il répond juste à 6 questions sur 7 dont la réponse est dans le CV, et il n'invente rien pour les 2 questions sans réponse.
- La qualité dépend d'abord du **retrieval**, et le retrieval dépend de la **préparation du document**. L'unique échec vient d'un PDF dont le texte est extrait dans le désordre : la description d'un projet est séparée de son titre. Avant d'indexer un CV, il vaudrait mieux nettoyer le texte (tabulations) et découper **par section ou par projet** plutôt qu'à taille fixe, pour que chaque chunk contienne un titre et sa description.
- Le **LLM juge** est utile pour automatiser l'évaluation, mais il faut le vérifier : il a noté 1/5 une réponse exacte et a donné trois notes différentes à la même abstention. Sa métrique doit être précisée, et il ne remplace pas une vérification de l'exactitude.
- Le code du prof contient une erreur sans conséquence ici, mais à corriger : `search_Kwargs={'k',5}` est ignoré et le retriever renvoie 4 chunks au lieu de 5.